In [1]:
import requests
import pandas as pd

# Coordinates for Luxembourg City
LAT = 49.6116
LON = 6.1319

# Same 30-day window you used for the bus data
START_DATE = "2025-07-24"
END_DATE = "2025-09-17"

# I focus on a few weather variables that might influence delays.
HOURLY_VARS = [
    "rain",          # rain (mm) in the previous hour
    "precipitation", # rain + snow + showers (mm)
    "weathercode"    # numeric weather condition code
]

# Build the API URL.
base_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": ",".join(HOURLY_VARS),
    "timezone": "Europe/Luxembourg"
}

print("Requesting hourly weather data from Open-Meteo...")
response = requests.get(base_url, params=params)
response.raise_for_status()  # raise error if something went wrong

data = response.json()

# The API returns an "hourly" section with parallel arrays.
hourly = data.get("hourly", {})

# I build a DataFrame from these arrays.
weather_df = pd.DataFrame(hourly)

# Convert the 'time' column to datetime.
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Just to be safe, keep only the columns I need.
cols_to_keep = ["time", "rain", "precipitation", "weathercode"]
weather_df = weather_df[cols_to_keep].copy()

# I create an hourly key that will be used later to merge with the bus data.
weather_df["time_hour"] = weather_df["time"]

print("Weather dataframe shape:", weather_df.shape)
print(weather_df.head())

# Save to CSV inside your project folder (adjust the path to your structure).
output_path = "../data/sample/weather_lux_hourly_2025-08-17_2025-09-16.csv"
weather_df.to_csv(output_path, index=False)

print(f"\nSaved weather data to: {output_path}")

Requesting hourly weather data from Open-Meteo...
Weather dataframe shape: (1344, 5)
                 time  rain  precipitation  weathercode           time_hour
0 2025-07-24 00:00:00   0.0            0.0            3 2025-07-24 00:00:00
1 2025-07-24 01:00:00   0.0            0.0            3 2025-07-24 01:00:00
2 2025-07-24 02:00:00   0.5            0.5           53 2025-07-24 02:00:00
3 2025-07-24 03:00:00   0.4            0.4           51 2025-07-24 03:00:00
4 2025-07-24 04:00:00   0.4            0.4           51 2025-07-24 04:00:00

Saved weather data to: ../data/sample/weather_lux_hourly_2025-08-17_2025-09-16.csv
